In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/oddrationale/mnist-in-csv/mnist_test.csv
/kaggle/input/datasets/oddrationale/mnist-in-csv/mnist_train.csv


# 📊 Principal Component Analysis (PCA) on MNIST

## 1. What is PCA?

**Principal Component Analysis (PCA)** is an **unsupervised dimensionality reduction technique** used to reduce the number of features in a dataset while preserving as much important information (variance) as possible.

PCA transforms the original features into a new set of features called **Principal Components (PCs)**.

### Why do we use PCA?

- Reduce the number of dimensions/features
- Remove redundant information
- Reduce computational complexity
- Make high-dimensional data easier to visualize
- Improve training efficiency in some machine learning problems

---

## 2. Why PCA for MNIST?

The MNIST dataset contains handwritten digits represented as **28 × 28 pixel images**.

Therefore:

**28 × 28 = 784 features**

Each image is converted into a vector containing 784 pixel values.

```text
Original MNIST Image
       ↓
    28 × 28
       ↓
  784 pixel features
       ↓
      PCA
       ↓
Reduced dimensions

In [3]:
import kagglehub

path = kagglehub.dataset_download("oddrationale/mnist-in-csv")

print("Dataset downloaded to:", path)

Dataset downloaded to: /kaggle/input/datasets/oddrationale/mnist-in-csv


In [4]:
import os

for dirname, _, filenames in os.walk(path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/oddrationale/mnist-in-csv/mnist_test.csv
/kaggle/input/datasets/oddrationale/mnist-in-csv/mnist_train.csv


In [6]:
import pandas as pd
import numpy as np

train_path = "/kaggle/input/datasets/oddrationale/mnist-in-csv/mnist_train.csv"

train_df = pd.read_csv(
    train_path,
    nrows=10000,
    dtype=np.uint8
)

print(train_df.shape)

(10000, 785)


In [7]:
print(train_df.shape)
print(train_df.columns[:10])
print(train_df.columns[-10:])

(10000, 785)
Index(['label', '1x1', '1x2', '1x3', '1x4', '1x5', '1x6', '1x7', '1x8', '1x9'], dtype='object')
Index(['28x19', '28x20', '28x21', '28x22', '28x23', '28x24', '28x25', '28x26',
       '28x27', '28x28'],
      dtype='object')


In [8]:
X = train_df.drop("label", axis=1)
y = train_df["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (10000, 784)
y shape: (10000,)


In [9]:
print(X.min().min())
print(X.max().max())

0
255


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Original shape:", X.shape)
print("Scaled shape:", X_scaled.shape)

Original shape: (10000, 784)
Scaled shape: (10000, 784)


In [11]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)

X_pca = pca.fit_transform(X_scaled)

print("PCA shape:", X_pca.shape)

PCA shape: (10000, 3)


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy_before = accuracy_score(y_test, y_pred)

print("Accuracy WITHOUT PCA:", accuracy_before)

Accuracy WITHOUT PCA: 0.876


In [15]:
from sklearn.decomposition import PCA

pca_100 = PCA(
    n_components=200,
    svd_solver="randomized",
    random_state=42
)

X_pca_100 = pca_100.fit_transform(X_scaled)

print("Original dimensions:", X_scaled.shape[1])
print("Reduced dimensions:", X_pca_100.shape[1])
print("Variance retained:", pca_100.explained_variance_ratio_.sum())

Original dimensions: 784
Reduced dimensions: 200
Variance retained: 0.9014125786392929


In [16]:
from sklearn.decomposition import PCA

pca_3 = PCA(
    n_components=3,
    svd_solver="randomized",
    random_state=42
)

X_pca_3 = pca_3.fit_transform(X_scaled)

print("Original shape:", X_scaled.shape)
print("PCA shape:", X_pca_3.shape)
print("Variance retained:", pca_3.explained_variance_ratio_.sum())

Original shape: (10000, 784)
PCA shape: (10000, 3)
Variance retained: 0.14575986633178636


In [17]:
pca_df = pd.DataFrame(
    X_pca_3,
    columns=["PC1", "PC2", "PC3"]
)

pca_df["Digit"] = y.values

pca_df.head()

,PC1,PC2,PC3,Digit
0,-0.997652,-4.652699,-0.740638,5
1,8.682758,-7.123700,-4.194685,0
2,2.439059,10.567253,-4.058946,4
3,-7.291486,-3.686136,3.065267,1
4,-4.791478,4.498237,-5.603298,9


In [20]:
import plotly.express as px

fig = px.scatter_3d(
    pca_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="Digit",
    title="MNIST Dataset - PCA 3D Visualization",
    labels={
        "PC1": "Principal Component 1",
        "PC2": "Principal Component 2",
        "PC3": "Principal Component 3",
        "Digit": "Digit"
    }
)

fig.update_traces(
    marker=dict(size=2)
)

fig.show()